In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import numpy as np

# veriyi oku
df = pd.read_csv('../data/processed/cleaned_tweets.csv')
df['tweet_created'] = pd.to_datetime(df['tweet_created'])

In [ ]:
# pasta grafik ile genel duygu dagilimi
# value_counts(): her kategoriden kac tane var hesaplar
# autopct: dilim icine yuzde yazar
plt.figure(figsize=(8, 8), dpi=70)
duygu_sayilari = df['airline_sentiment'].value_counts()
renkler = ['#ff9999','#66b3ff','#99ff99']
plt.pie(duygu_sayilari, labels=duygu_sayilari.index, autopct='%1.1f%%',
        startangle=140, colors=renkler)
plt.title('Havayolu Tweetleri: Duygu Dagilimi', fontsize=16)
plt.savefig('grafik1_duygu_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# tum temizlenmis tweet metinlerini tek bir string olarak birlestir
# WordCloud bu uzun stringdeki kelimelerin sikligina gore boyutlandirma yapar
tum_metin = " ".join(tweet for tweet in df['cleaned_text'].dropna())

plt.figure(figsize=(10, 6), dpi=70)
wc = WordCloud(width=800, height=400, background_color='white',
               max_words=100).generate(tum_metin)
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Tweetlerde En Cok Gecen Kelimeler', fontsize=16)
plt.savefig('grafik2_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# gun bazli gruplama yaparak her gun kac tweet atilmis hesapliyorum
# dt.date ile datetime'dan sadece tarih kismini aliyorum
df['tarih_gun'] = df['tweet_created'].dt.date
gunluk = df.groupby('tarih_gun').size()

plt.figure(figsize=(12, 5), dpi=70)
sns.lineplot(x=gunluk.index, y=gunluk.values, marker='o', color='b')
plt.title('Gunlere Gore Tweet Sayisi', fontsize=14)
plt.xlabel('Tarih')
plt.ylabel('Tweet Sayisi')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('grafik3_zaman_serisi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# en cok tweet atan 10 kullaniciyi bul ve bar grafik ciz
# value_counts(): her kullanicinin kac tweet attigini sayar
# head(10): en basta olan 10 tanesini al
plt.figure(figsize=(10, 6), dpi=70)
aktif = df['name'].value_counts().head(10)
sns.barplot(x=aktif.values, y=aktif.index, palette='viridis')
plt.title('En Cok Tweet Atan 10 Kullanici', fontsize=14)
plt.xlabel('Tweet Sayisi')
plt.ylabel('Kullanici')
plt.savefig('grafik4_aktif_kullanicilar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# sadece negatif ve sebebi belli olanlari filtrele
# "Belirtilmedi" olanlari disarida birakiyorum cunku bunlar pozitif/notr tweetler
negatif = df[(df['airline_sentiment'] == 'negative') & (df['negativereason'] != 'Belirtilmedi')]

plt.figure(figsize=(12, 6), dpi=70)
sns.countplot(data=negatif, y='negativereason',
              order=negatif['negativereason'].value_counts().index,
              palette='rocket')
plt.title('Negatif Tweetlerin Sebepleri', fontsize=14)
plt.xlabel('Sikayet Sayisi')
plt.ylabel('Sebep')
plt.savefig('grafik5_sikayet_nedenleri.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# her havayolu icin pozitif/negatif/notr tweet sayilarini hesapla
# crosstab: iki kategorik degisken arasindaki frekanslari cikarir
# normalize='index': satirlara gore yuzdelik hesaplar (her havayolunun toplami %100)
ct = pd.crosstab(df['airline'], df['airline_sentiment'], normalize='index') * 100

# istersem mutlak sayilari da gosterebilirim
ct_mutlak = pd.crosstab(df['airline'], df['airline_sentiment'])

plt.figure(figsize=(12, 6), dpi=70)
ct.plot(kind='bar', stacked=True, color=['#ff9999','#66b3ff','#99ff99'],
        figsize=(12, 6))
plt.title('Havayolu Bazli Duygu Dagilimi (Yuzdelik)', fontsize=14)
plt.xlabel('Havayolu')
plt.ylabel('Yuzde (%)')
plt.legend(title='Duygu', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('grafik6_havayolu_duygu.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMutlak sayilar:")
print(ct_mutlak)

In [ ]:
# her havayolu icin her sikayet sebebinden kac adet var — crosstab ile hesapla
# sadece negatif tweetleri al ve sebebi belli olanlari filtrele
negatif_df = df[(df['airline_sentiment'] == 'negative') & (df['negativereason'] != 'Belirtilmedi')]
ct_hm = pd.crosstab(negatif_df['airline'], negatif_df['negativereason'])

# heatmap: renk yogunluguyla sayilari gorsel olarak gosterir
# annot=True: hucrelerin icine sayilari yazar
# cmap='YlOrRd': sari-turuncu-kirmizi renk skalasi (dusuk=sari, yuksek=kirmizi)
# fmt='d': sayilari tam sayi olarak goster (ondalik yok)
plt.figure(figsize=(14, 6), dpi=70)
sns.heatmap(ct_hm, annot=True, cmap='YlOrRd', fmt='d', linewidths=0.5)
plt.title('Havayolu x Sikayet Sebebi Isi Haritasi', fontsize=14)
plt.xlabel('Sikayet Sebebi')
plt.ylabel('Havayolu')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('grafik7_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()